# 01. Annual maximum series, independence and trend

This notebook builds the annual maximum series (AMS) of daily rainfall at the Bangladesh Water Development Board gauge at Khulna (station CL510, 1961 to 2019) and tests it for serial independence and monotonic trend. 

**Input**

- `data/rainfall/khulna_CL510_daily.csv`: daily rainfall (columns `Date` and rainfall in mm), or a pre-computed annual series (columns `Year` and rainfall in mm)



## Configuration

In [ ]:
import math
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import stats

ROOT = Path("..")                      # repository root; notebooks are run from notebooks/
INPUT_FILE = ROOT / "data" / "rainfall" / "khulna_CL510_daily.csv"
OUTDIR = ROOT / "results" / "01_trend"
OUTDIR.mkdir(parents=True, exist_ok=True)

## 1. Annual maximum series

The AMS is the largest daily rainfall of each calendar year. The reader accepts a daily record, from which the annual maxima are extracted, or a series that already holds one value per year. Years without data are omitted and are not set to zero.

In [ ]:
def load_ams(path):
    """Read the rainfall file and return the annual maximum series (Year, Rainfall_mm)."""
    df = pd.read_csv(path)
    df.columns = [col.strip() for col in df.columns]

    rain_col = next((col for col in df.columns
                     if any(k in col.lower() for k in ["rain", "precip", "pcp"])),
                    df.columns[-1])

    if any(col.lower() == "date" for col in df.columns):
        dcol = next(col for col in df.columns if col.lower() == "date")
        df[dcol] = pd.to_datetime(df[dcol], errors="coerce")
        df = df.dropna(subset=[dcol])
        df["Year"] = df[dcol].dt.year
        ams = df.groupby("Year")[rain_col].max().reset_index()
        ams.columns = ["Year", "Rainfall_mm"]
        print(f"Daily record: annual maxima extracted for {len(ams)} years.")
    else:
        ycol = "Year" if "Year" in df.columns else df.columns[0]
        ams = df[[ycol, rain_col]].copy()
        ams.columns = ["Year", "Rainfall_mm"]
        print(f"Annual series: {len(ams)} years supplied.")

    ams = ams.dropna().sort_values("Year").reset_index(drop=True)
    ams["Rainfall_mm"] = ams["Rainfall_mm"].astype(float)
    return ams


ams = load_ams(INPUT_FILE)
ams.to_csv(OUTDIR / "AMS.csv", index=False)
ams.head(10)

## 2. Descriptive statistics

The skewness is the bias-corrected sample skewness.

In [ ]:
x = ams["Rainfall_mm"].values

summary = {
    "Years of record":  len(x),
    "First year":       int(ams["Year"].min()),
    "Last year":        int(ams["Year"].max()),
    "Mean (mm)":        round(np.mean(x), 2),
    "Median (mm)":      round(np.median(x), 2),
    "Std deviation (mm)": round(np.std(x, ddof=1), 2),
    "Skewness":         round(stats.skew(x, bias=False), 3),
    "Minimum (mm)":     round(np.min(x), 1),
    "Maximum (mm)":     round(np.max(x), 1),
    "Year of maximum":  int(ams.loc[np.argmax(x), "Year"]),
}

desc = pd.DataFrame(list(summary.items()), columns=["Statistic", "Value"])
desc.to_csv(OUTDIR / "descriptive_statistics.csv", index=False)
desc

## 3. Serial independence

Positive autocorrelation inflates the variance of the Mann-Kendall statistic and can produce a spurious trend (Yue et al. 2002), so independence is checked first. The Durbin-Watson statistic (Durbin and Watson 1950) is computed from the residuals $e_i$ of an ordinary least-squares fit of the AMS against year,

$$DW = \frac{\sum_{i=2}^{n}\left(e_i - e_{i-1}\right)^2}{\sum_{i=1}^{n} e_i^2}$$

and values between 1.5 and 2.5 are taken to indicate no first-order autocorrelation. The lag-1 autocorrelation coefficient $r_1$ is tested against zero with the large-sample two-sided p-value $p = 2\left[1 - \Phi\left(|r_1|\sqrt{n}\right)\right]$.

In [ ]:
def durbin_watson(residuals):
    r = np.asarray(residuals, dtype=float)
    return float(np.sum(np.diff(r) ** 2) / np.sum(r ** 2))


yr = ams["Year"].values
n  = len(x)

slope, intercept, *_ = stats.linregress(yr, x)
resid = x - (slope * yr + intercept)

dw     = durbin_watson(resid)
lag1   = float(pd.Series(x).autocorr(lag=1))
lag1_p = 2 * (1 - stats.norm.cdf(abs(lag1) * math.sqrt(n)))

dw_verdict = ("within 1.5 to 2.5: no first-order autocorrelation"
              if 1.5 < dw < 2.5 else
              "outside 1.5 to 2.5: first-order autocorrelation indicated")

auto = pd.DataFrame([
    ["Durbin-Watson statistic", round(dw, 3), dw_verdict],
    ["Lag-1 autocorrelation", round(lag1, 4),
     f"p = {lag1_p:.3f} ({'no' if lag1_p > 0.05 else 'some'} serial correlation)"],
], columns=["Test", "Value", "Interpretation"])
auto.to_csv(OUTDIR / "autocorrelation_check.csv", index=False)
auto

## 4. Trend tests

Monotonic trend is tested with the Mann-Kendall test (Mann 1945; Kendall 1975) at a significance level of 0.05. The statistic is

$$S = \sum_{k=1}^{n-1}\sum_{j=k+1}^{n}\mathrm{sgn}\left(x_j - x_k\right)$$

with the tie-corrected variance

$$\mathrm{Var}(S) = \frac{n(n-1)(2n+5) - \sum_{q=1}^{g} t_q\left(t_q-1\right)\left(2t_q+5\right)}{18}$$

and the standardised statistic $Z = (S \mp 1)/\sqrt{\mathrm{Var}(S)}$ for $S \gtrless 0$. The trend magnitude is the Sen (1968) slope, the median of all pairwise slopes; an ordinary least-squares slope is reported alongside it.

In [ ]:
def mann_kendall(x):
    x = np.asarray(x, dtype=float)
    n = len(x)
    s = sum(np.sum(np.sign(x[k + 1:] - x[k])) for k in range(n - 1))
    _, counts = np.unique(x, return_counts=True)
    ties = counts[counts > 1]
    tie_term = np.sum(ties * (ties - 1) * (2 * ties + 5))
    var_s = (n * (n - 1) * (2 * n + 5) - tie_term) / 18.0
    if   s > 0: z = (s - 1) / math.sqrt(var_s)
    elif s < 0: z = (s + 1) / math.sqrt(var_s)
    else:       z = 0.0
    p = 2 * (1 - stats.norm.cdf(abs(z)))
    return s, z, p


def sens_slope(years, x):
    years = np.asarray(years, float); x = np.asarray(x, float)
    slopes = []
    for i in range(len(x) - 1):
        dt = years[i + 1:] - years[i]
        dv = x[i + 1:] - x[i]
        slopes.extend(dv[dt != 0] / dt[dt != 0])
    return float(np.median(slopes))


slope_lr, intercept, r_val, p_lr, _ = stats.linregress(yr, x)
s, z, p_mk = mann_kendall(x)
sen = sens_slope(yr, x)

mk_verdict = ("significant trend (p < 0.05)"
              if p_mk < 0.05 else
              "no significant trend (p >= 0.05)")

trend = pd.DataFrame([
    ["Linear regression slope (mm/yr)", round(slope_lr, 4), f"p = {p_lr:.4f}"],
    ["Mann-Kendall S", int(s), "test statistic S"],
    ["Mann-Kendall Z", round(z, 4), "standardised statistic Z"],
    ["Mann-Kendall p-value", round(p_mk, 4), mk_verdict],
    ["Sen's slope (mm/yr)", round(sen, 4), "median of pairwise slopes"],
], columns=["Test", "Value", "Interpretation"])
trend.to_csv(OUTDIR / "trend_tests.csv", index=False)
trend